# 03 — Business Analysis & Insights
Key questions answered with Looker-ready charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/books_clean.csv')
print(df.shape)

## Q1. Which formats dominate and how do their prices compare?

In [ ]:
fig = px.box(df, x='format', y='price_usd', color='category',
    title='Price Distribution by Format & Category')
fig.show()

## Q2. Does a higher price mean better ratings?

In [ ]:
r, p = stats.pearsonr(df['price_usd'], df['rating'])
print(f'Pearson r = {r:.3f}, p = {p:.4f}')
fig = px.scatter(df, x='price_usd', y='rating', color='format',
    size='reviews', hover_data=['title','author'],
    title=f'Price vs Rating (r={r:.2f})', opacity=0.6)
fig.show()

## Q3. Top publishers — who owns the best-seller list?

In [ ]:
top_pub = df.groupby('publisher').agg(count=('rank','count'), avg_rating=('rating','mean')).sort_values('count', ascending=False).head(12)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_pub.index, top_pub['count'], color='steelblue')
ax2 = ax.twiny()
ax2.plot(top_pub['avg_rating'], top_pub.index, 'o-', color='tomato', linewidth=1.5)
ax.set_xlabel('Titles on List')
ax2.set_xlabel('Avg Rating', color='tomato')
plt.title('Top 12 Publishers')
plt.tight_layout()
plt.show()

## Q4. What sub-genres have the most review volume?

In [ ]:
genre_agg = df.groupby('sub_genre').agg(avg_reviews=('reviews','mean'), count=('rank','count')).query('count >= 3').sort_values('avg_reviews', ascending=False).head(12)
fig = px.bar(genre_agg.reset_index(), x='avg_reviews', y='sub_genre',
    orientation='h', color='avg_reviews', color_continuous_scale='Blues',
    title='Avg Reviews by Sub-Genre (min 3 books)')
fig.show()

## Q5. Publication era vs longevity on list

In [ ]:
pivot = df.pivot_table(values='weeks_on_list', index='pub_era', columns='category', aggfunc='mean')
plt.figure(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Avg Weeks on List — Publication Era × Category')
plt.tight_layout()
plt.show()

## Q6. Most engaged sub-genres (engagement score)

In [ ]:
top_eng = df.groupby('sub_genre')['engagement_score'].mean().sort_values(ascending=False).head(12)
fig = px.bar(top_eng.reset_index(), x='engagement_score', y='sub_genre',
    orientation='h', color='engagement_score', color_continuous_scale='Viridis',
    title='Avg Engagement Score by Sub-Genre (log reviews × rating)')
fig.show()